# moe-injection-interp: Colab quickstart

1. **Runtime → Change runtime type → GPU** (A100 or L4 recommended; a T4 has 15 GB and no native bf16, which is tight).
2. Run the cells in order. Everything uses the same `moe_interp` package and CLI as a laptop or SSH machine.

Results can go to Google Drive so they survive disconnects. Resume an interrupted run with `--resume`.

In [ ]:
# --- configuration -----------------------------------------------------------------------------
REPO_URL = "https://github.com/<your-username>/moe-injection-interp.git"   # <- change me
USE_DRIVE = True                    # save results to Google Drive
EXPERIMENT = "smoke_test"


In [ ]:
import os, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/moe_interp_results"
else:
    OUT_ROOT = "/content/results"
os.makedirs(OUT_ROOT, exist_ok=True)
OUTPUT = f"{OUT_ROOT}/{EXPERIMENT}"
print("results ->", OUTPUT)

In [ ]:
%cd /content
!test -d moe-injection-interp || git clone {REPO_URL} moe-injection-interp
%cd /content/moe-injection-interp
!git log -1 --oneline
!pip install -q -e .
# The working directory is the repo root, so `import moe_interp` works without restarting the runtime.

In [ ]:
# Optional: an HF token gives faster, rate-limit-free downloads (Colab: key icon -> secrets -> HF_TOKEN)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
!python -m moe_interp.cli doctor --model-id allenai/OLMoE-1B-7B-0125-Instruct

**Verify extraction on the real model.** This loads OLMoE (≈14 GB download on first use) and checks router hooks, dispatched weights, and residual bookkeeping. Every check should say `PASS`.

In [ ]:
!python -m moe_interp.cli verify --config configs/olmoe.yaml --json-out {OUTPUT}_verify.json

**Run the smoke-test experiment** (generation + routing/activation traces), then compute metrics and plots.

In [ ]:
!python -m moe_interp.cli run --config configs/olmoe.yaml --data data/examples.jsonl --output {OUTPUT} --resume --analyze

In [ ]:
from pathlib import Path
from IPython.display import Image, display
for p in sorted(Path(OUTPUT, "plots").glob("*.png")):
    print(p.name); display(Image(filename=str(p), width=700))

In [ ]:
import pandas as pd
gens = pd.read_json(f"{OUTPUT}/generations.jsonl", lines=True)
gens[["sample_id", "condition", "num_prompt_tokens", "num_generated_tokens", "response_text"]]

For deeper analysis, open `notebooks/analysis_demo.ipynb` with `MOE_INTERP_RESULTS` set to `OUTPUT`. You can also download the results folder and analyze it on a laptop CPU.

Optional GPU integration test:
```
!MOE_INTERP_RUN_INTEGRATION=1 python -m pytest tests/integration -v -s
```